# Solana (SOL) Technical Analysis & On-Chain Signals Dashboard


In [ ]:
# Install dependencies
!pip install -q requests pandas matplotlib numpy


# Solana (SOL) Technical Analysis & On-Chain Signals Dashboard

We will:
1. Fetch 6-month SOL/USD price history
2. Calculate RSI, EMA crossovers, and support/resistance levels
3. Visualize price with technical indicators
4. Pull on-chain activity metrics
5. Summarize bullish vs. bearish signals

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import requests

plt.style.use('seaborn-v0_8-darkgrid')
print('Libraries imported successfully.')

## Step 1: Fetch SOL/USD Historical Price Data

We'll use the CoinGecko free API to retrieve 180 days of daily SOL price data.

In [ ]:
# Fetch SOL price data from CoinGecko API (free, no key required)
url = 'https://api.coingecko.com/api/v3/coins/solana/market_chart'
params = {
    'vs_currency': 'usd',
    'days': '180',
    'interval': 'daily'
}

response = requests.get(url, params=params)
data = response.json()

prices = data['prices']
df = pd.DataFrame(prices, columns=['timestamp', 'price'])
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
df['date'] = df['timestamp'].dt.date
df = df[['date', 'price']].reset_index(drop=True)

print(f'Fetched {len(df)} days of SOL price data')
print(df.head())
print(f"\nPrice range: ${df['price'].min():.2f} - ${df['price'].max():.2f}")

## Step 2: Compute Technical Indicators

Calculate RSI (14-period), EMA (20 and 50-period), MACD, and identify support/resistance levels.

In [ ]:
def calculate_rsi(prices, period=14):
    delta = prices.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

def calculate_ema(prices, period):
    return prices.ewm(span=period, adjust=False).mean()

# Calculate indicators
df['rsi_14'] = calculate_rsi(df['price'], 14)
df['ema_20'] = calculate_ema(df['price'], 20)
df['ema_50'] = calculate_ema(df['price'], 50)
df['sma_200'] = df['price'].rolling(window=200).mean()

# Identify support and resistance (last 60 days)
recent = df.tail(60)
support = recent['price'].min()
resistance = recent['price'].max()
midpoint = (support + resistance) / 2

print(f'Support Level (60-day low): ${support:.2f}')
print(f'Resistance Level (60-day high): ${resistance:.2f}')
print(f'Midpoint: ${midpoint:.2f}')
print(f'\nCurrent Price: ${df["price"].iloc[-1]:.2f}')
print(f'RSI (14): {df["rsi_14"].iloc[-1]:.2f}')
print(f'EMA 20 vs EMA 50: {"BULLISH CROSS" if df["ema_20"].iloc[-1] > df["ema_50"].iloc[-1] else "BEARISH CROSS"}')

## Step 3: Visualize Price Action with Technical Indicators

Plot SOL price with RSI, EMA crossovers, and support/resistance zones.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Price chart with EMAs and support/resistance
ax1 = axes[0]
ax1.plot(df['date'], df['price'], label='SOL Price', linewidth=2, color='#00D4AA')
ax1.plot(df['date'], df['ema_20'], label='EMA 20', alpha=0.7, color='orange')
ax1.plot(df['date'], df['ema_50'], label='EMA 50', alpha=0.7, color='purple')
ax1.axhline(y=support, color='red', linestyle='--', linewidth=1.5, label=f'Support: ${support:.2f}')
ax1.axhline(y=resistance, color='green', linestyle='--', linewidth=1.5, label=f'Resistance: ${resistance:.2f}')
ax1.fill_between(df['date'], support, resistance, alpha=0.1, color='gray')
ax1.set_ylabel('Price (USD)', fontsize=11)
ax1.set_title('SOL/USD Price Action with EMA & Support/Resistance (180 days)', fontsize=12, fontweight='bold')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# RSI chart
ax2 = axes[1]
ax2.plot(df['date'], df['rsi_14'], label='RSI (14)', color='#1f77b4', linewidth=2)
ax2.axhline(y=70, color='red', linestyle='--', alpha=0.5, label='Overbought (70)')
ax2.axhline(y=30, color='green', linestyle='--', alpha=0.5, label='Oversold (30)')
ax2.fill_between(df['date'], 30, 70, alpha=0.1, color='blue')
ax2.set_ylabel('RSI', fontsize=11)
ax2.set_xlabel('Date', fontsize=11)
ax2.set_title('Relative Strength Index (RSI 14)', fontsize=12, fontweight='bold')
ax2.set_ylim(0, 100)
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Price action chart generated.')

## Step 4: Pull Basic On-Chain Activity Metrics

Fetch summary on-chain data from a public endpoint to complement technical analysis.

In [ ]:
# Fetch on-chain metrics from Solscan/CoinGecko APIs
# CoinGecko provides basic on-chain summaries without auth

try:
    coin_data = requests.get('https://api.coingecko.com/api/v3/coins/solana').json()
    
    market_cap = coin_data['market_data']['market_cap']['usd']
    trading_volume = coin_data['market_data']['total_volume']['usd']
    circulating_supply = coin_data['market_data']['circulating_supply']
    ath = coin_data['market_data']['ath']['usd']
    current_price = coin_data['market_data']['current_price']['usd']
    distance_from_ath = ((current_price - ath) / ath) * 100
    
    print('\n=== SOLANA ON-CHAIN & MARKET METRICS ===')
    print(f'Current Price: ${current_price:.2f}')
    print(f'Market Cap: ${market_cap:,.0f}')
    print(f'24h Trading Volume: ${trading_volume:,.0f}')
    print(f'Circulating Supply: {circulating_supply:,.0f} SOL')
    print(f'All-Time High: ${ath:.2f}')
    print(f'Distance from ATH: {distance_from_ath:.1f}%')
    print(f'Volume/Market Cap Ratio: {(trading_volume / market_cap):.3f}')
    
except Exception as e:
    print(f'Error fetching on-chain data: {e}')

## Step 5: Summary — Recovery Signals vs. Weakness Indicators

Consolidate technical and on-chain findings to assess SOL's current state.

In [ ]:
current_price = df['price'].iloc[-1]
current_rsi = df['rsi_14'].iloc[-1]
ema_20 = df['ema_20'].iloc[-1]
ema_50 = df['ema_50'].iloc[-1]

print('\n=== TECHNICAL ANALYSIS SUMMARY ===')
print('\nBULLISH SIGNALS:')
bullish_count = 0
if current_price > support:
    print(f'✓ Price above support (${support:.2f})')
    bullish_count += 1
if ema_20 > ema_50:
    print(f'✓ EMA 20 above EMA 50 (bullish crossover trend)')
    bullish_count += 1
if current_rsi < 70:
    print(f'✓ RSI not overbought ({current_rsi:.1f} < 70)')
    bullish_count += 1
if current_price > df['price'].iloc[-30]:  # Higher than 30 days ago
    print(f'✓ Price higher than 30 days ago')
    bullish_count += 1

print('\nBEARISH SIGNALS:')
bearish_count = 0
if current_price < resistance:
    print(f'✗ Price below resistance (${resistance:.2f})')
    bearish_count += 1
if ema_20 < ema_50:
    print(f'✗ EMA 20 below EMA 50 (downtrend)')
    bearish_count += 1
if current_rsi > 50:
    print(f'✗ RSI elevated ({current_rsi:.1f} > 50, moderate strength only)')
    bearish_count += 1
if df['price'].iloc[-1] < df['sma_200'].iloc[-1]:
    print(f'✗ Price below 200-day SMA (long-term downtrend)')
    bearish_count += 1

print(f'\nNet Signal: {bullish_count} bullish vs {bearish_count} bearish indicators')
if bullish_count > bearish_count:
    print('→ RECOVERY BIAS (technical setup favors upside)')
elif bearish_count > bullish_count:
    print('→ CAUTION (technical setup favors downside)')
else:
    print('→ NEUTRAL (balanced risk/reward)')

---

## Reference

[https://protraderdaily.com](https://protraderdaily.com/crypto/is-solana-dead-technical-analysis-recovery-signs)
